In [12]:
"""
Step 2: controlled experiments on how to merge CICIDS2018 into CICIDS2017.
 
Same model family as newmain.ipynb (RF + XGBoost soft vote, RobustScaler,
balanced sample weights) but smaller so each run takes minutes.
Every experiment retrains from scratch; the scaler is fitted on its own train set.
 
EXPERIMENTS
  E0  Baseline - 2017 only
        Reproduces the gap you saw.
  E1  2017 + 2018 attacks only
        Your original idea.
  E2  2017 + 2018 attacks + matched 2018 benign
        Adds benign 2018 rows (BENIGN_RATIO x attack rows) so "2018 = attack" is not a shortcut.
  E3  E2 minus shifted features
        Does removing the features that differ most between 2017 and 2018 benign help
        once 2018 data is already in training?
  E4  2017 + duplicated 2017 attacks (record-count control)
        Adds as many attack rows as E1 but copied from 2017. Separates "more records"
        from "records from a different network".
  E5  2017 only minus shifted features (feature-shift control)
        Does removing the shifted features alone fix the 2017 -> 2018 gap, with no 2018 data?
  E6  Leave-one-variant-out (one run per 2018 attack variant)
        Trains like E2 but withholds ONE attack variant completely (all its rows), then tests
        on it. Examples: train FTP-BruteForce, test SSH-Bruteforce; train GoldenEye + Slowloris
        + Hulk, test SlowHTTPTest. The strictest generalization test here.
  E7  Alert threshold on P(Normal)
        Instead of waiting for Normal to lose the argmax, raise an alert whenever
        P(Normal) < t. Reports, per 2018 variant, how many attack flows would be
        flagged and how many benign flows would be falsely flagged. Run on the E0 and
        E2 models, and also added to every E6 row for the withheld variant.
 
TEST SETS
  2017 test   random 10% of a capped 2017 sample (in-distribution check only)
  2018 test   the LAST part of every (Source File, Raw Label) block, with a gap before it,
              so neighbouring near-duplicate flows do not leak into training
  E6 test     every row of the withheld variant (capped by E6_TEST_CAP)
"""
import warnings
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
 
warnings.filterwarnings('ignore')
 

In [14]:
 
# ---------------- CONFIG ----------------
PATH_2017 = r'Preprocessing/cicids2017_preprocessed.csv'
PATH_2018 = r'Preprocessing/cicids2018_extra.csv'
RUN = ['E0', 'E1', 'E2', 'E3', 'E4', 'E5', 'E6', 'E7']   # delete entries to skip experiments
CAP_2017 = 60000        # max rows per class taken from 2017 (speed)
CAP_2018 = 20000        # max rows per 2018 attack class used for training
BENIGN_RATIO = 1.0      # 2018 benign rows = ratio x 2018 attack rows (E2, E3, E6)
TEST_FRAC = 0.30        # last 30% of each 2018 block is test
GAP_FRAC = 0.02         # 2% of each block discarded between train and test
DROP_K = 5              # E3 / E5: remove this many most shifted features
E6_TEST_CAP = 50000     # max rows of a withheld variant used for testing
MIN_VARIANT_ROWS = 500  # E6 ignores variants with fewer rows than this
E7_THRESHOLDS = [0.5, 0.9, 0.99]   # alert when P(Normal) < t
E7_REPORT_T = 0.9       # threshold used for the 'correct class among alerts' column
N_TREES = 100           # trees for RF and XGBoost (lower = faster)
SEED = 42
# ----------------------------------------
 
CLASSES = ['Brute Force', 'DDoS', 'DoS', 'Normal Traffic', 'Port Scan']
META = ['Attack Type', 'Raw Label', 'Source File']
NORMAL = 'Normal Traffic'
 
NAMES = {
    'E0': 'E0 Baseline: 2017 only',
    'E1': 'E1 2017 + 2018 attacks only',
    'E2': 'E2 2017 + 2018 attacks + matched 2018 benign',
    'E3': f'E3 E2 minus top-{DROP_K} shifted features',
    'E4': 'E4 2017 + duplicated 2017 attacks (record-count control)',
    'E5': f'E5 2017 only minus top-{DROP_K} shifted features',
}
NAMES_E7 = {'E0': NAMES['E0'], 'E2': NAMES['E2']}
 
d18 = pd.read_csv(PATH_2018)
FEATURES = [c for c in d18.columns if c not in META]
d17 = pd.read_csv(PATH_2017, usecols=FEATURES + ['Attack Type'])
print(f'2017: {d17.shape}   2018: {d18.shape}   features: {len(FEATURES)}')
 
 
def cap_per_class(df, n):
    out = [g.sample(min(len(g), n), random_state=SEED) for _, g in df.groupby('Attack Type')]
    return pd.concat(out)
 
 
def block_split(df):
    tr, te = [], []
    for _, g in df.groupby(['Source File', 'Raw Label'], sort=False):
        n = len(g)
        cut = int(n * (1 - TEST_FRAC))
        gap = int(n * GAP_FRAC)
        tr.append(g.iloc[:cut])
        te.append(g.iloc[cut + gap:])
    return pd.concat(tr), pd.concat(te)

2017: (2515963, 41)   2018: (3999579, 43)   features: 40


In [15]:
 
# 2017: capped sample, random 90/10 split
s17 = cap_per_class(d17, CAP_2017)
tr17, te17 = train_test_split(s17, test_size=0.10, stratify=s17['Attack Type'], random_state=SEED)
 
# 2018: block split, then separate attack and benign training rows
tr18, te18 = block_split(d18)
tr18_att_all = tr18[tr18['Attack Type'] != NORMAL]
tr18_att = cap_per_class(tr18_att_all, CAP_2018)
tr18_ben_all = tr18[tr18['Attack Type'] == NORMAL]
te18_ben = te18[te18['Attack Type'] == NORMAL]
 
print('\n2018 train attack rows used:\n', tr18_att['Attack Type'].value_counts().to_string())
print('\n2018 TEST rows:\n', te18['Attack Type'].value_counts().to_string())
 
# ---- feature shift: 2017 benign vs 2018 benign ----
b17 = tr17[tr17['Attack Type'] == NORMAL]
b17 = b17.sample(min(20000, len(b17)), random_state=SEED)
b18 = tr18_ben_all.sample(min(20000, len(tr18_ben_all)), random_state=SEED)
ks = pd.Series({f: ks_2samp(b17[f], b18[f]).statistic for f in FEATURES}).sort_values(ascending=False)
print('\nMost shifted features (KS statistic, 2017 benign vs 2018 benign; 1.0 = totally different):')
print(ks.head(12).round(3).to_string())
shifted = ks.head(DROP_K).index.tolist()
keep = [f for f in FEATURES if f not in shifted]
 
 
def fit(train, feats):
    """Train RF + XGBoost on `train`, return a function df -> predicted labels."""
    le = LabelEncoder().fit(CLASSES)
    y = le.transform(train['Attack Type'])
    sc = RobustScaler().fit(train[feats].values)
    X = sc.transform(train[feats].values)
    w = compute_sample_weight('balanced', y)
 
    rf = RandomForestClassifier(n_estimators=N_TREES, max_depth=20, max_features=0.4,
                                n_jobs=-1, random_state=SEED).fit(X, y, sample_weight=w)
    xg = XGBClassifier(n_estimators=N_TREES, max_depth=6, learning_rate=0.1, subsample=0.8,
                       colsample_bytree=0.5, tree_method='hist', random_state=SEED,
                       n_jobs=-1).fit(X, y, sample_weight=w)
 
    def predict(df):
        Xs = sc.transform(df[feats].values)
        P = (rf.predict_proba(Xs) + xg.predict_proba(Xs)) / 2
        return le.classes_[P.argmax(axis=1)]
 
    def proba(df):
        Xs = sc.transform(df[feats].values)
        return (rf.predict_proba(Xs) + xg.predict_proba(Xs)) / 2
 
    predict.proba = proba
    predict.classes = list(le.classes_)
    return predict
 
 
def alert_columns(predict, df, true_class):
    """E7 numbers for one set of flows: alert rate at each threshold and, for attacks,
    how often the top non-Normal class is the right one among alerted flows."""
    P = predict.proba(df)
    ni = predict.classes.index(NORMAL)
    pn = P[:, ni]
    out = {f'alert P(Normal)<{t}': (pn < t).mean() for t in E7_THRESHOLDS}
    if true_class != NORMAL:
        Pn = P.copy()
        Pn[:, ni] = -1
        top = np.array(predict.classes)[Pn.argmax(axis=1)]
        alerted = pn < E7_REPORT_T
        out[f'correct class among alerts @{E7_REPORT_T}'] = (top[alerted] == true_class).mean() if alerted.any() else np.nan
    return out
 
 
def evaluate(predict, train, feats, name):
    row = {'experiment': name, 'train_rows': len(train), 'n_features': len(feats)}
    row['2017 macroF1'] = f1_score(te17['Attack Type'], predict(te17), average='macro')
    p18 = predict(te18)
    t18 = te18['Attack Type'].values
    present = [c for c in CLASSES if c in set(t18)]
    for c in present:
        row[f'2018 recall {c}'] = recall_score(t18 == c, p18 == c)
    row['2018 macroF1'] = f1_score(t18, p18, labels=present, average='macro')
    return row
 
 
def benign_sample(n_att):
    n = int(BENIGN_RATIO * n_att)
    return tr18_ben_all.sample(min(n, len(tr18_ben_all)), random_state=SEED)
 
 
def build_dup():
    counts = tr18_att['Attack Type'].value_counts()
    att17 = tr17[tr17['Attack Type'].isin(counts.index)]
    extra = [g.sample(counts[c], replace=True, random_state=SEED)
             for c, g in att17.groupby('Attack Type')]
    return pd.concat([tr17] + extra)
 
 


2018 train attack rows used:
 Attack Type
Brute Force    20000
DDoS           20000
DoS            20000

2018 TEST rows:
 Attack Type
Normal Traffic    982359
DDoS               56167
DoS                55024
Brute Force        26346

Most shifted features (KS statistic, 2017 benign vs 2018 benign; 1.0 = totally different):
Fwd IAT Min              0.410
Fwd Packet Length Min    0.394
Bwd IAT Min              0.390
Min Packet Length        0.387
Bwd Packet Length Std    0.371
Fwd Packet Length Std    0.366
Bwd IAT Max              0.358
Bwd IAT Total            0.357
min_seg_size_forward     0.356
Fwd Packet Length Max    0.353
Bwd Packet Length Max    0.341
Fwd IAT Std              0.340


In [16]:
# ---- E0 to E5 ----
trains = {
    'E0': (tr17, FEATURES),
    'E1': (pd.concat([tr17, tr18_att]), FEATURES),
    'E2': (pd.concat([tr17, tr18_att, benign_sample(len(tr18_att))]), FEATURES),
    'E3': (pd.concat([tr17, tr18_att, benign_sample(len(tr18_att))]), keep),
    'E4': (build_dup(), FEATURES),
    'E5': (tr17, keep),
}
rows = []
models = {}
for key in ['E0', 'E1', 'E2', 'E3', 'E4', 'E5']:
    if key not in RUN:
        continue
    train, feats = trains[key]
    print(f'\nRunning {NAMES[key]} ...')
    predict = fit(train, feats)
    if key in NAMES_E7:
        models[key] = predict
    rows.append(evaluate(predict, train, feats, NAMES[key]))
 
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 50)
if rows:
    res = pd.DataFrame(rows).set_index('experiment')
    print('\nRESULTS E0-E5 (2018 columns = held-out 2018 blocks; recall of Normal = 1 - false alarm rate)')
    print(res.round(3).T.to_string())
    res.to_csv('experiment_results.csv')
    print('Shifted features removed in E3 and E5:', shifted)



Running E0 Baseline: 2017 only ...

Running E1 2017 + 2018 attacks only ...

Running E2 2017 + 2018 attacks + matched 2018 benign ...

Running E3 E2 minus top-5 shifted features ...

Running E4 2017 + duplicated 2017 attacks (record-count control) ...

Running E5 2017 only minus top-5 shifted features ...

RESULTS E0-E5 (2018 columns = held-out 2018 blocks; recall of Normal = 1 - false alarm rate)
experiment                  E0 Baseline: 2017 only  E1 2017 + 2018 attacks only  E2 2017 + 2018 attacks + matched 2018 benign  E3 E2 minus top-5 shifted features  E4 2017 + duplicated 2017 attacks (record-count control)  E5 2017 only minus top-5 shifted features
train_rows                              224235.000                   284235.000                                    344235.000                          344235.000                                                284235.000                                 224235.000
n_features                                  40.000                      

In [17]:

# ---- E6: leave-one-variant-out ----
if 'E6' in RUN:
    rows6 = []
    variants = d18[d18['Attack Type'] != NORMAL].groupby('Attack Type')['Raw Label'].unique()
    for cls, labs in variants.items():
        labs = [l for l in labs if (d18['Raw Label'] == l).sum() >= MIN_VARIANT_ROWS]
        if len(labs) < 2:
            print(f'\nE6 skipped for {cls}: fewer than two usable 2018 variants')
            continue
        for lab in labs:
            held = d18[d18['Raw Label'] == lab]
            held = held.sample(min(len(held), E6_TEST_CAP), random_state=SEED)
            att = cap_per_class(tr18_att_all[tr18_att_all['Raw Label'] != lab], CAP_2018)
            train = pd.concat([tr17, att, benign_sample(len(att))])
            seen = sorted(set(att.loc[att['Attack Type'] == cls, 'Raw Label']))
            print(f'\nRunning E6 leave-out {lab} (trained on {seen}) ...')
            predict = fit(train, FEATURES)
            p = predict(held)
            pb = predict(te18_ben)
            row6 = {
                'experiment': f'E6 leave-out {lab}',
                'true class': cls,
                'class variants seen in training': ', '.join(seen),
                'unseen rows tested': len(held),
                'recall (predicted correct class)': (p == cls).mean(),
                'predicted Normal (missed)': (p == NORMAL).mean(),
                'predicted other attack class': ((p != cls) & (p != NORMAL)).mean(),
                '2018 benign recall': (pb == NORMAL).mean(),
            }
            if 'E7' in RUN:
                for k, v in alert_columns(predict, held, cls).items():
                    row6[f'E7 unseen variant: {k}'] = v
                for k, v in alert_columns(predict, te18_ben, NORMAL).items():
                    row6[f'E7 benign false alarm: {k}'] = v
            rows6.append(row6)
    if rows6:
        r6 = pd.DataFrame(rows6).set_index('experiment')
        print('\nRESULTS E6 (each row: a 2018 attack variant the model never saw in training)')
        print(r6.round(3).T.to_string())
        r6.to_csv('experiment_e6_results.csv')
 


E6 skipped for Brute Force: fewer than two usable 2018 variants

Running E6 leave-out DDOS attack-LOIC-UDP (trained on ['DDOS attack-HOIC']) ...

Running E6 leave-out DDOS attack-HOIC (trained on ['DDOS attack-LOIC-UDP']) ...

Running E6 leave-out DoS attacks-GoldenEye (trained on ['DoS attacks-Hulk', 'DoS attacks-SlowHTTPTest', 'DoS attacks-Slowloris']) ...

Running E6 leave-out DoS attacks-Slowloris (trained on ['DoS attacks-GoldenEye', 'DoS attacks-Hulk', 'DoS attacks-SlowHTTPTest']) ...

Running E6 leave-out DoS attacks-Hulk (trained on ['DoS attacks-GoldenEye', 'DoS attacks-SlowHTTPTest', 'DoS attacks-Slowloris']) ...

RESULTS E6 (each row: a 2018 attack variant the model never saw in training)
experiment                                         E6 leave-out DDOS attack-LOIC-UDP E6 leave-out DDOS attack-HOIC                                 E6 leave-out DoS attacks-GoldenEye                                 E6 leave-out DoS attacks-Slowloris                                          

In [18]:
# ---- E7: alert threshold on P(Normal), applied to the E0 and E2 models ----
if 'E7' in RUN:
    rows7 = []
    for key, predict in models.items():
        for lab, g in te18.groupby('Raw Label'):
            if len(g) < 20:
                continue
            true = g['Attack Type'].iloc[0]
            row7 = {'model': NAMES[key], 'Raw Label': lab, 'true class': true, 'rows': len(g)}
            if true != NORMAL:
                row7['argmax recall'] = (predict(g) == true).mean()
            row7.update(alert_columns(predict, g, true))
            rows7.append(row7)
    if rows7:
        r7 = pd.DataFrame(rows7).set_index(['model', 'Raw Label'])
        print('\nRESULTS E7 (held-out 2018 blocks; for the Benign rows the alert columns are FALSE ALARM rates)')
        print(r7.round(3).to_string())
        r7.to_csv('experiment_e7_results.csv')
    else:
        print('\nE7 needs E0 and/or E2 in RUN - nothing to report.')
 
print('\nDone.')


RESULTS E7 (held-out 2018 blocks; for the Benign rows the alert columns are FALSE ALARM rates)
                                                                        true class    rows  alert P(Normal)<0.5  alert P(Normal)<0.9  alert P(Normal)<0.99  argmax recall  correct class among alerts @0.9
model                                        Raw Label                                                                                                                                                    
E0 Baseline: 2017 only                       Benign                 Normal Traffic  982359                0.005                0.086                 0.309            NaN                              NaN
                                             DDOS attack-HOIC                 DDoS   55682                0.000                0.000                 0.819          0.000                              NaN
                                             DDOS attack-LOIC-UDP             DDoS     485  